# CSE 5830 - Homework 2
Naive Bayes spam/ham classifier

Skeleton only — fill in the TODOs yourself. Per the assignment: toolboxes are fine for text preprocessing, but the Naive Bayes training/classification logic must be your own code.

In [182]:
import re
import zipfile
import random

## 1. Load data
Download `smsspamcollection.zip` from https://archive.ics.uci.edu/dataset/228/sms+spam+collection and place it in this folder.

In [183]:
def load_sms_data(path_to_zip):
    """
    Read the smsspamcollection.zip file and return a list of (label, text) tuples.
    label is 'spam' or 'ham'.
    """
    # TODO: open the zip, read SMSSpamCollection, parse each line into (label, text)
    
    results = []
    
    with  zipfile.ZipFile(path_to_zip) as z:
        with z.open("SMSSpamCollection") as f:
            for raw_line in f:
                text = raw_line.decode("utf-8").strip()
                a, b = text.split("\t", 1)
                results.append((a, b))

    return results

    

## 2. Preprocessing

In [184]:
def tokenize(text):
    """
    Split text into normalized word tokens:
      - split on whitespace/punctuation
      - lowercase
      - link related word forms (e.g. simple stemming/lemmatization)
    Return a list of tokens.
    """
    # TODO: implement (ok to use a toolbox here, e.g. re, nltk stemmer)
    words = re.findall(r"[a-zA-Z']+", text.lower())
    from nltk.stem import PorterStemmer
    tokens = []
    for word in words:
        stemmer = PorterStemmer()
        tokens.append(stemmer.stem(word))
    return tokens

## 3. Train/test split

In [185]:
def train_test_split(data, train_frac=0.8, seed=None):
    """
    Shuffle `data` and split into (train, test) using train_frac for training.
    """
    # TODO: implement
    from sklearn.model_selection import train_test_split as sk_split
    split = sk_split(data, train_size=train_frac, random_state=seed)
    return split



## 4. Training — build the Naive Bayes model
This is the core algorithm — implement it yourself, no shortcuts.

Steps:
1. tokenize every message
2. count word frequencies separately within spam messages and ham messages
3. convert counts to P(word|class) for each class
4. compute priors P(spam) and P(ham) from class sizes
5. return everything needed for classification

In [186]:
def train_naive_bayes(train_data):
    """
    Given a list of (label, text) tuples, build and return a model containing:
      - P(word | spam) for every word seen
      - P(word | ham) for every word seen
      - class priors P(spam), P(ham)
      - counts of training examples per class (for the results printout)
    """
    # TODO: implement
    spam_word_counts = {}   # word -> total occurrences in spam messages
    ham_word_counts = {}    # word -> total occurrences in ham messages
    spam_total_words = 0    # sum of all counts in spam_word_counts
    ham_total_words = 0     # sum of all counts in ham_word_counts
    spam_message_count = 0
    ham_message_count = 0

    for label, text in train_data:
        tokens = tokenize(text)
        if label == "spam":
            word_counts = spam_word_counts
            # ... which total/count variables go with spam?
        else:
            word_counts = ham_word_counts
            # ... which total/count variables go with ham?

        for word in tokens:
            count = word_counts.get(word, 0)
            word_counts[word] = count + 1
            # TODO: also bump the matching total-word counter

            if label == "spam":
                spam_total_words += 1
            else:
                ham_total_words += 1
            
        if label == "spam":
            spam_message_count += 1
        else:
            ham_message_count += 1 
    # TODO: bump the matching message counter (once per message, not per word)


    p_word_given_spam = {}
    p_word_given_ham = {}
    # TODO: walk spam_word_counts / ham_word_counts and convert each
    #   raw count into a probability (count / total for that class).
    for word, count in spam_word_counts.items():
        p_word_given_spam[word] = count / spam_total_words


    total_messages = spam_message_count + ham_message_count
    
    for word, count in ham_word_counts.items():
        p_word_given_ham[word] = count / ham_total_words

    # TODO: compute p_spam, p_ham from the message counts above

    model = {
        "p_word_given_spam": p_word_given_spam,
        "p_word_given_ham": p_word_given_ham,
        "p_spam": spam_message_count / total_messages,   # TODO
        "p_ham": ham_message_count / total_messages,    # TODO
        "spam_count": spam_message_count,
        "ham_count": ham_message_count,
    }
    return model

    

## 5. Classification
Must use LOG PROBABILITIES (sum of logs) instead of multiplying raw probabilities, to avoid floating-point underflow. Any word in the message that never appeared in training should be ignored.

Steps:
1. tokenize the text
2. compute log P(spam) + sum(log P(word|spam)) for words seen in training
3. compute log P(ham)  + sum(log P(word|ham))  for words seen in training
4. return whichever class has the higher score

In [187]:
import math 
def classify(model, text):
    """
    Given a trained model and a raw text message, return predicted label
    ('spam' or 'ham').
    
    """
    # TODO: implement
    tokens = tokenize(text)
    spam_score = math.log(model["p_spam"])
    ham_score = math.log(model["p_ham"])

    for word in tokens:
        if word in model["p_word_given_spam"]:
            spam_score += math.log(model["p_word_given_spam"][word])

        if word in model["p_word_given_ham"]:
            ham_score += math.log(model["p_word_given_ham"][word])


    return "spam" if spam_score > ham_score else "ham" 


## 6. Evaluation

In [188]:
def evaluate(model, test_data):
    ham_correct = 0
    ham_total = 0
    spam_correct = 0
    spam_total = 0

    for label, text in test_data:
        prediction = classify(model, text)
        # TODO: based on `label`, bump the right _total counter, and bump
        #   the matching _correct counter if `prediction == label`
        if label == "ham":
          ham_total += 1
          if prediction == label:
             ham_correct += 1
        else:
            spam_total += 1
            if prediction == label:
                spam_correct += 1

            


    results = {
        "ham_correct_pct": ham_correct / ham_total * 100,
        "spam_correct_pct": spam_correct / spam_total * 100,
        "total_accuracy": (ham_correct + spam_correct) / (ham_total + spam_total) * 100,
        "false_positive": (ham_total - ham_correct) / ham_total * 100,
    }
    return results


In [189]:
def print_results(train_data, results, out_file=None):
    """
    Print results in the exact format the assignment requires, and optionally
    write/append to results.txt.
    """
    ham_count = sum(1 for label, _ in train_data if label == "ham")
    spam_count = sum(1 for label, _ in train_data if label == "spam")

    lines = [
        f"Size of ham training set: {ham_count}",
        f"Size of spam training set: {spam_count}",
        f"Percentage of ham classified correctly: {results['ham_correct_pct']:.1f}",
        f"Percentage of spam classified correctly: {results['spam_correct_pct']:.1f}",
        f"Total accuracy: {results['total_accuracy']:.2f}",
        f"False Positive: {results['false_positive']:.2f}",
    ]
    for line in lines:
        print(line)
    if out_file:
        with open(out_file, "a") as f:
            f.write("\n".join(lines) + "\n\n")

## 7. Main

In [190]:
data = load_sms_data("sms+spam+collection.zip")
train_data, test_data = train_test_split(data, train_frac=0.8, seed=42)

model = train_naive_bayes(train_data)

In [191]:
# Run 1: train on train_data, test on train_data
results_on_train = evaluate(model, train_data)
print_results(train_data, results_on_train, out_file="results.txt")

Size of ham training set: 3873
Size of spam training set: 586
Percentage of ham classified correctly: 15.1
Percentage of spam classified correctly: 32.9
Total accuracy: 17.47
False Positive: 84.87


In [192]:
# Run 2: train on train_data, test on test_data
results_on_test = evaluate(model, test_data)
print_results(train_data, results_on_test, out_file="results.txt")

Size of ham training set: 3873
Size of spam training set: 586
Percentage of ham classified correctly: 19.2
Percentage of spam classified correctly: 47.8
Total accuracy: 23.32
False Positive: 80.82


## 8. Use of AI

For this assignment I used AI to help me understand and get hints about the assignment. I also used it to help me read code by helping set up a skeleton notebook and then work on the assignment on my own.